# BoneNano step 1 — geometry, angles, dark/flat

Writes everything into `codedHT.h5` **except** the projection data, which is a
95 GB pass and lives in [`step2_preprocess.ipynb`](step2_preprocess.ipynb).

Unlike `AtomiumS2_coded/step1`, there is no format conversion to do — the
DanMAX orca file is already HDF5. So this step is light: read the geometry and
the rotation angles, build the dark, flat and dead-pixel mask, and write the
small datasets.

Scan 86 is CA1 + CA2 + the **rotating sample**, 4001 frames. The rotation
angle is not in the orca file; it is in the companion `scan-0086.h5` at
`/entry/instrument/tom_ry/value`, in degrees.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../../src'))
import numpy as np, cupy as cp, h5py
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

def cbshow(ax, img, title, cmap='gray', **kw):
    im = ax.imshow(np.asarray(img), cmap=cmap, **kw)
    ax.set_title(title, fontsize=9)
    cax = make_axes_locatable(ax).append_axes('right', size='5%', pad=0.05)
    ax.figure.colorbar(im, cax=cax)

print('GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode())

## Config

In [ ]:
# --- everything comes from config_step6.conf. Edit THAT file, not this cell.
from holotomocupy.config import parse_args_coded
CFG = parse_args_coded('config_step6.conf')

RAW_DIR = '/data3/cekmekci/bone_nano_holotomo'   # raw_dir; parse_args_coded has no key for it
OUT_H5  = CFG.in_file

DARK_H5  = f'{RAW_DIR}/scan-0067_orca.h5'   # dark
FLAT_H5  = f'{RAW_DIR}/scan-0068_orca.h5'   # flat: no CA1, no CA2
GRID_H5  = f'{RAW_DIR}/scan-0086_orca.h5'   # CA1 + CA2 + rotating sample, 95 GB
META_H5  = f'{RAW_DIR}/scan-0086.h5'        # companion: rotation angle
ORCA     = '/entry/instrument/orca/data'
THETA_DS = '/entry/instrument/tom_ry/value'   # DEGREES

ENERGY  = CFG.energy
Z1      = CFG.z1                 # focus -> SAMPLE
Z_CODE  = CFG.z1_code            # focus -> CA2
Z_DET   = CFG.focusdetectordistance  # focus -> detector  (TODO: confirm per beamtime)
DET_PIX = CFG.detector_pixelsize # UNBINNED
nz, n   = CFG.crop_nz, CFG.crop_n   # NATIVE crop resolution -- NOT CFG.nz/CFG.n,
                                    # which are step 6's own BINNED solve grid.
                                    # Steps 1-5 never bin anything; that is
                                    # CodedReader's job alone at step 6 (same
                                    # as AtomiumS2_coded). See the README's
                                    # "steps 1-5 never bin" section.
CROP    = (nz, n)
DEZINGER = (3, 0.8)
PREVIEW_BIN = CFG.bin   # cosmetic ONLY -- shrinks a notebook thumbnail in
                        # PDATA_CHECK below. Not used in any registration or
                        # writing math; see the note below.

# NOT read for any registration/writing math: CFG.ntheta/start_theta/stride
# (preprocessing never subsamples the angles -- see GEOM_THETA below) or
# CFG.bin/CFG.nz/CFG.n (preprocessing never bins -- see above). All four are
# the step6.py/CodedReader read-time parameters, not preprocessing's.

print(f'{ENERGY} keV   z1(sample) {Z1} m   z1_code(CA2) {Z_CODE} m   z_det {Z_DET} m')
print(f'crop {CROP}, native resolution -- step 6 bins to {CFG.nz} x {CFG.n} at bin={CFG.bin}')
assert Z_CODE < Z1, 'CA2 must be BEFORE the sample for RecCoded'

## Preprocessing helpers

In [ ]:
# Ports of phascade.preprocess.{dark_subtract_clip, fill_dead, remove_outliers,
# flat_field_divide}. Identical to this folder's own step0a_convert -- same
# detector, same chain. The two stages MUST preprocess the same way, or the
# warm start from stage 0 does not line up with the data here.
from cupyx.scipy.ndimage import median_filter

def dark_subtract_clip(data, dark_mean):
    out = np.asarray(data, 'float32') - dark_mean[np.newaxis]
    np.clip(out, 0, None, out=out)
    return out

def _runs(idx):
    return [] if len(idx) == 0 else np.split(idx, np.where(np.diff(idx) > 1)[0] + 1)

def fill_dead(data, mask):
    out = np.array(data, copy=True)
    ny, nx = mask.shape
    for g in _runs(np.where(mask.mean(1) > 0.5)[0]):
        lo, hi = g[0] - 1, g[-1] + 1
        hl, hh = lo >= 0, hi < ny
        for r in g:
            if hl and hh:
                w = (r - lo) / (hi - lo)
                out[..., r, :] = (1 - w) * out[..., lo, :] + w * out[..., hi, :]
            elif hl:
                out[..., r, :] = out[..., lo, :]
            elif hh:
                out[..., r, :] = out[..., hi, :]
    for g in _runs(np.where(mask.mean(0) > 0.5)[0]):
        lo, hi = g[0] - 1, g[-1] + 1
        hl, hh = lo >= 0, hi < nx
        for c in g:
            if hl and hh:
                w = (c - lo) / (hi - lo)
                out[..., :, c] = (1 - w) * out[..., :, lo] + w * out[..., :, hi]
            elif hl:
                out[..., :, c] = out[..., :, lo]
            elif hh:
                out[..., :, c] = out[..., :, hi]
    return out

def remove_outliers(data, radius=3, threshold=0.8):
    # window is radius x radius, NOT 2*radius+1 -- matches phascade
    res = np.asarray(data, 'float32').copy()
    for k in range(res.shape[0]):
        f = cp.asarray(res[k])
        med = median_filter(f, [radius, radius])
        res[k] = cp.asnumpy(cp.where(cp.abs(f - med) > med * threshold, med, f))
    return res

def flat_field_divide(frames, flat_mean):
    fmean = float(np.asarray(flat_mean).mean()) + 1e-12
    return np.nan_to_num(np.asarray(frames, 'float32') / fmean, nan=1.0)

def bin2(a, nbin):
    for _ in range(nbin):
        a = 0.5 * (a[..., 0::2, :] + a[..., 1::2, :])
        a = 0.5 * (a[..., :, 0::2] + a[..., :, 1::2])
    return a

## Geometry and the rotation angles

`CodedReader` reads `/exchange/theta` in **degrees** and converts, so that is
what gets written.

The angle check below matters: `tom_ry` should rise monotonically from 0 to
360. A flat stretch, a jump, or a count that disagrees with the frame count
means the scan and the metadata are out of step, and every projection after
that point is attributed to the wrong angle.

In [ ]:
with h5py.File(GRID_H5, 'r') as f:
    NANG, NY, NX = f[ORCA].shape
cy, cx = NY // 2, NX // 2
SL = (slice(cy - CROP[0] // 2, cy + CROP[0] // 2),
      slice(cx - CROP[1] // 2, cx + CROP[1] // 2))

with h5py.File(META_H5, 'r') as f:
    theta_all = np.asarray(f[THETA_DS], 'float64')
assert len(theta_all) == NANG, f'{len(theta_all)} angles vs {NANG} frames'

# ALL frames, always -- no angle subsampling at preprocessing time. Quick-look
# runs subsample later, at step 6, via config_step6.conf's own
# ntheta/start_theta/stride (CodedReader indexes directly into this fully
# stored array). Same convention AtomiumS2_coded uses: write everything once,
# select a subset only when solving.
NTHETA = NANG
sel = np.arange(NANG)
theta_deg = theta_all[sel]

mag       = Z_DET / Z1
voxelsize = DET_PIX / mag   # NATIVE voxel size -- no bin factor, steps 1-5 never bin

print(f'orca {NY} x {NX}, {NANG} frames; crop rows {SL[0].start}:{SL[0].stop} '
      f'cols {SL[1].start}:{SL[1].stop}')
print(f'theta: ALL {NTHETA} angles converted, [{theta_deg.min():.3f}, {theta_deg.max():.3f}] deg, '
      f'step {np.diff(theta_deg).mean():.4f} deg')
print(f'voxel {voxelsize*1e9:.2f} nm   magnification {mag:.3f}')
print('quick-look subsampling, if wanted, happens at step 6 via '
      "config_step6.conf's own ntheta/start_theta/stride")

plt.figure(figsize=(5, 3))
plt.plot(theta_all, '.', ms=2)
plt.xlabel('frame'); plt.ylabel('tom_ry [deg]'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Dark, flat, dead-pixel mask

The same dark (scan 67) and flat (scan 68) as this folder's `step0a`,
processed the same way. They have to match: stage 0's `code.npy` and `prb.npy` were fitted
on data normalised like this, so a different normalisation here would make the
warm start inconsistent with the data it is supposed to explain.

Written into `codedHT.h5` so step 2 reads them back, which is how
`AtomiumS2_coded/step1` does it — one file, no sidecar. The difference is that
it copies the raw dark and white *stacks* in; here only the processed mean
dark, flat and dead-mask are stored, because the raw scan is 95 GB and nothing
else consumes the stacks.

In [ ]:
def _mean_stack(path, sl):
    with h5py.File(path, 'r') as f:
        d = f[ORCA]
        acc = np.zeros((sl[0].stop - sl[0].start, sl[1].stop - sl[1].start), 'float64')
        for i in range(d.shape[0]):
            acc += d[i, sl[0], sl[1]]
        return (acc / d.shape[0]).astype('float32')

dark = _mean_stack(DARK_H5, SL)
flat = dark_subtract_clip(_mean_stack(FLAT_H5, SL)[None], dark)[0]

thr = 0.05 * np.median(flat[flat > 0])
dead_mask = flat < thr
print(f'dead threshold {thr:.1f} ADU   dead pixels {int(dead_mask.sum())}')
print('  dead rows', np.where(dead_mask.mean(1) > 0.5)[0].tolist())
print('  dead cols', np.where(dead_mask.mean(0) > 0.5)[0].tolist())

flat = fill_dead(flat[None], dead_mask)[0]
if DEZINGER:
    flat = remove_outliers(flat[None], *DEZINGER)[0]
flat_mean = float(flat.mean())
flat_norm = (flat / flat_mean).astype('float32')

# Into the HDF5, as AtomiumS2_coded/step1 does -- one file, no sidecar.
# The difference from that pipeline: it copies the raw dark and white STACKS
# in, because its raw scan is small enough. Here the raw scan is 95 GB, so
# only the PROCESSED mean dark, flat and dead-mask are stored. Nothing else
# consumes the stacks.
os.makedirs(os.path.dirname(OUT_H5), exist_ok=True)
with h5py.File(OUT_H5, 'a') as f:
    for k, v in (('dark', dark), ('flat_norm', flat_norm),
                 ('dead_mask', dead_mask.astype('uint8'))):
        if f'/exchange/{k}' in f:
            del f[f'/exchange/{k}']
        f.create_dataset(f'/exchange/{k}', data=v)
    if '/exchange/flat_mean' in f:
        del f['/exchange/flat_mean']
    f.create_dataset('/exchange/flat_mean', data=np.array([flat_mean], 'float64'))
print(f'flat_mean {flat_mean:.1f} ADU -> /exchange/{{dark,flat_norm,dead_mask,flat_mean}}')
print('   steps 2 and 3 read them back from there')

fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
cbshow(ax[0], dark, 'dark (mean)')
cbshow(ax[1], flat_norm, 'flat: dark-sub -> dead-fix -> dezinger -> /mean')
cbshow(ax[2], dead_mask.astype('float32'), 'dead_mask')
plt.tight_layout(); plt.show()

## Write the small datasets

In [ ]:
with h5py.File(OUT_H5, 'a') as f:
    for k, v in (('energy', ENERGY), ('z1', Z1),
                 ('focusdetectordistance', Z_DET),
                 ('detector_pixelsize', DET_PIX),
                 ('voxelsize', voxelsize)):
        if f'/exchange/{k}' in f:
            del f[f'/exchange/{k}']
        f.create_dataset(f'/exchange/{k}', data=np.array([v], 'float64'))
    # CodedReader reads theta in DEGREES and converts.
    if '/exchange/theta' in f:
        del f['/exchange/theta']
    f.create_dataset('/exchange/theta', data=theta_deg.astype('float32'))
print('wrote geometry +', len(theta_deg), 'angles ->', OUT_H5)
print('NOTE detector_pixelsize is written UNBINNED; CodedReader scales it by 2**bin.')
print('Next: step2_preprocess.ipynb')